In [1]:
# Q4 - Financial profile

# Question: is a customer's financial profile (housing loan, personal loan, credit default) linked to saying yes?

# Each group is compared with the overall yes rate (the Q1 baseline) and shown with its customer count. Groups under 100 customers are flagged and not interpreted. "unknown" is kept as its own group (decision 1).

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q4, step 1: yes rate by housing loan, personal loan and credit default
subscribed = clean_df["subscribed_term_deposit"] == "yes"
baseline_rate = subscribed.mean() * 100

financial_columns = ["housing_loan", "personal_loan", "credit_default"]
value_order = ["yes", "no", "unknown"]

summaries = []
for column in financial_columns:
    summary = (
        pd.DataFrame({"value": clean_df[column], "subscribed": subscribed})
        .groupby("value")["subscribed"]
        .agg(customers="count", subscribers="sum")
        .reindex(value_order)
    )
    summary["yes_rate_pct"] = (summary["subscribers"] / summary["customers"] * 100).round(2)
    summary["vs_baseline_points"] = (summary["yes_rate_pct"] - baseline_rate).round(2)
    summary["share_of_customers_pct"] = (summary["customers"] / len(clean_df) * 100).round(2)
    summary["share_of_subscribers_pct"] = (summary["subscribers"] / subscribed.sum() * 100).round(2)
    summary["small_group"] = summary["customers"] < 100
    summaries.append(summary)

    print(f"{column}: customers {summary['customers'].sum()} (should be {len(clean_df)}), "
          f"subscribers {summary['subscribers'].sum()} (should be {subscribed.sum()})")

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print()
pd.concat(summaries, keys=financial_columns, names=["column", "value"])

housing_loan: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
personal_loan: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
credit_default: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
Baseline yes rate: 11.27%



customers  subscribers  yes_rate_pct  \
column         value                                           
housing_loan   yes          21571         2507         11.62   
               no           18615         2025         10.88   
               unknown        990          107         10.81   
personal_loan  yes           6248          683         10.93   
               no           33938         3849         11.34   
               unknown        990          107         10.81   
credit_default yes              3            0          0.00   
               no           32577         4196         12.88   
               unknown       8596          443          5.15   

                        vs_baseline_points  share_of_customers_pct  \
column         value                                                 
housing_loan   yes                    0.35                   52.39   
               no                    -0.39                   45.21   
               unknown               -0.46                    2.40   
personal_loan  yes                   -0.34                   15.17   
               no                     0.07                   82.42   
               unknown               -0.46                    2.40   
credit_default yes                  -11.27                    0.01   
               no                     1.61                   79.12   
               unknown               -6.12                   20.88   

                        share_of_subscribers_pct  small_group  
column         value                                           
housing_loan   yes                         54.04        False  
               no                          43.65        False  
               unknown                      2.31        False  
personal_loan  yes                         14.72        False  
               no                          82.97        False  
               unknown                      2.31        False  
credit_default yes                          0.00         True  
               no                          90.45        False  
               unknown                      9.55        False

In [4]:
# Q4, step 2: observed vs expected yes rate, given each group's age mix
age_edges = [0, 25, 30, 35, 40, 45, 50, 55, 60, 65, 120]
age_labels = ["under 25", "25-29", "30-34", "35-39", "40-44",
              "45-49", "50-54", "55-59", "60-64", "65 and over"]
age_band = pd.cut(clean_df["age"], bins=age_edges, labels=age_labels, right=False)

def observed_vs_expected(column):
    rows = []
    for group in value_order:
        in_group = clean_df[column] == group

        # Yes rate of customers in ALL OTHER groups, within each age band
        other_rates = (subscribed[~in_group]
                       .groupby(age_band[~in_group], observed=True).mean()
                       .reindex(age_labels))
        group_counts = age_band[in_group].value_counts().reindex(age_labels).fillna(0)

        assert not ((group_counts > 0) & other_rates.isna()).any(), f"no comparison customers for {column} = {group}"
        expected_subscribers = (group_counts * other_rates).sum()

        customers = in_group.sum()
        observed_subscribers = subscribed[in_group].sum()
        observed_rate = observed_subscribers / customers * 100
        expected_rate = expected_subscribers / customers * 100
        margin = 1.96 * (expected_rate / 100 * (1 - expected_rate / 100) / customers) ** 0.5 * 100

        rows.append({
            "value": group,
            "customers": customers,
            "observed_subscribers": observed_subscribers,
            "observed_yes_rate_pct": round(observed_rate, 2),
            "expected_yes_rate_pct": round(expected_rate, 2),
            "difference_points": round(observed_rate - expected_rate, 2),
            "chance_margin_points": round(margin, 2),
            "small_group": customers < 100,
        })

    result = pd.DataFrame(rows).set_index("value")
    result["outside_chance_margin"] = result["difference_points"].abs() > result["chance_margin_points"]
    print(f"{column}: customers {result['customers'].sum()} (should be {len(clean_df)}), "
          f"observed subscribers {result['observed_subscribers'].sum()} (should be {subscribed.sum()})")
    return result

financial_vs_expected = pd.concat(
    [observed_vs_expected(column) for column in financial_columns],
    keys=financial_columns, names=["column", "value"],
)

print()
financial_vs_expected

housing_loan: customers 41176 (should be 41176), observed subscribers 4639 (should be 4639)
personal_loan: customers 41176 (should be 41176), observed subscribers 4639 (should be 4639)
credit_default: customers 41176 (should be 41176), observed subscribers 4639 (should be 4639)



customers  observed_subscribers  \
column         value                                      
housing_loan   yes          21571                  2507   
               no           18615                  2025   
               unknown        990                   107   
personal_loan  yes           6248                   683   
               no           33938                  3849   
               unknown        990                   107   
credit_default yes              3                     0   
               no           32577                  4196   
               unknown       8596                   443   

                        observed_yes_rate_pct  expected_yes_rate_pct  \
column         value                                                   
housing_loan   yes                      11.62                  10.99   
               no                       10.88                  11.45   
               unknown                  10.81                  11.38   
personal_loan  yes                      10.93                  11.25   
               no                       11.34                  10.96   
               unknown                  10.81                  11.38   
credit_default yes                       0.00                   8.87   
               no                       12.88                   6.16   
               unknown                   5.15                  11.85   

                        difference_points  chance_margin_points  small_group  \
column         value                                                           
housing_loan   yes                   0.63                  0.42        False   
               no                   -0.57                  0.46        False   
               unknown              -0.57                  1.98        False   
personal_loan  yes                  -0.32                  0.78        False   
               no                    0.39                  0.33        False   
               unknown              -0.57                  1.98        False   
credit_default yes                  -8.87                 32.17         True   
               no                    6.72                  0.26        False   
               unknown              -6.70                  0.68        False   

                        outside_chance_margin  
column         value                           
housing_loan   yes                       True  
               no                        True  
               unknown                  False  
personal_loan  yes                      False  
               no                        True  
               unknown                  False  
credit_default yes                      False  
               no                        True  
               unknown                   True

In [ ]:
# Q4, step 3: do unknown and recorded credit default customers differ in how and when they were contacted?

status_order = ["unknown", "recorded"]
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]


frame = pd.DataFrame({
    "credit_status": clean_df["credit_default"].where(clean_df["credit_default"] == "unknown", "recorded"),
    "contact_type": clean_df["contact_type"],
    "contact_month": clean_df["contact_month"],
    "subscribed": subscribed,
})


status_totals = (
    frame.groupby("credit_status")["subscribed"]
    .agg(customers="count", subscribers="sum")
    .reindex(status_order)
)
print(status_totals)
print(f"Customers: {status_totals['customers'].sum()} (should be {len(clean_df)}), "
      f"subscribers: {status_totals['subscribers'].sum()} (should be {subscribed.sum()})")
print()


def mix_and_rates(column, order):
    grouped = frame.groupby([column, "credit_status"])["subscribed"].agg(customers="count", subscribers="sum")
    customers = (grouped["customers"].unstack("credit_status")
                 .reindex(index=order, columns=status_order).fillna(0).astype(int))
    subscribers = (grouped["subscribers"].unstack("credit_status")
                   .reindex(index=order, columns=status_order).fillna(0).astype(int))
    rates = (subscribers / customers * 100).where(customers >= 100).round(2)
    mix = (customers / customers.sum() * 100).round(1)

    result = pd.DataFrame({
        "unknown_customers": customers["unknown"],
        "recorded_customers": customers["recorded"],
        "pct_of_unknown_customers": mix["unknown"],
        "pct_of_recorded_customers": mix["recorded"],
        "unknown_yes_rate_pct": rates["unknown"],
        "recorded_yes_rate_pct": rates["recorded"],
    })
    result["gap_points"] = (result["unknown_yes_rate_pct"] - result["recorded_yes_rate_pct"]).round(2)
    print(f"{column}: customers {customers.values.sum()} (should be {len(clean_df)}), "
          f"unknown {customers['unknown'].sum()}, recorded {customers['recorded'].sum()}")
    return result


contact_summary = mix_and_rates("contact_type", ["cellular", "telephone"])
month_summary = mix_and_rates("contact_month", month_order)
print()
print("By contact type (yes rates blank where a group has under 100 customers):")
print(contact_summary)
print()
print("By month (yes rates blank where a group has under 100 customers):")
print(month_summary)
print()


# Expected yes rate for unknown customers if they behaved like recorded customers contacted the same way
def expected_unknown_subscribers(keys):
    recorded_rates = frame[frame["credit_status"] == "recorded"].groupby(keys)["subscribed"].mean()
    unknown_counts = frame[frame["credit_status"] == "unknown"].groupby(keys).size()
    assert len(unknown_counts.index.difference(recorded_rates.index)) == 0, f"no recorded customers for {keys}"
    return (unknown_counts * recorded_rates.reindex(unknown_counts.index)).sum()

unknown_customers = status_totals.loc["unknown", "customers"]
unknown_subscribers = status_totals.loc["unknown", "subscribers"]
observed_rate = unknown_subscribers / unknown_customers * 100


rows = []
for label, keys in [("contact type", ["contact_type"]),
                    ("month", ["contact_month"]),
                    ("contact type and month", ["contact_type", "contact_month"])]:
    expected_subscribers = expected_unknown_subscribers(keys)
    rows.append({
        "allowing for": label,
        "observed_subscribers": unknown_subscribers,
        "expected_subscribers": round(expected_subscribers, 1),
        "observed_yes_rate_pct": round(observed_rate, 2),
        "expected_yes_rate_pct": round(expected_subscribers / unknown_customers * 100, 2),
    })


print("Unknown credit default customers: observed vs expected if they behaved like recorded customers:")
print(pd.DataFrame(rows).set_index("allowing for"))

               customers  subscribers
credit_status                        
unknown             8596          443
recorded           32580         4196
Customers: 41176 (should be 41176), subscribers: 4639 (should be 4639)

contact_type: customers 41176 (should be 41176), unknown 8596, recorded 32580
contact_month: customers 41176 (should be 41176), unknown 8596, recorded 32580

By contact type (yes rates blank where a group has under 100 customers):
              unknown_customers  recorded_customers  pct_of_unknown_customers  \
contact_type                                                                    
cellular                   4363               21772                      50.8   
telephone                  4233               10808                      49.2   

              pct_of_recorded_customers  unknown_yes_rate_pct  \
contact_type                                                    
cellular                           66.8                  6.67   
telephone                